# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [9]:
%pip install --upgrade plotly nbformat ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.8/649.8 kB 11.1 MB/s  0:00:00
  Attempting uninstall: ipython
    Found existing installation: ipython 9.12.0
    Uninstalling ipython-9.12.0:
      Successfully uninstalled ipython-9.12.0
  Attempting uninstall: nbformat━━━━━━━━━━━━━━━━━━━━━━ 0/2 [ipython]
    Found existing installation: nbformat 5.10.4━━ 0/2 [ipython]
    Uninstalling nbformat-5.10.4:━━━━━━━━━━━━━━━━━ 0/2 [ipython]
      Successfully uninstalled nbformat-5.10.4━━━━ 0/2 [ipython]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nbformat]━━ 1/2 [nbformat]

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import plotly.io as pio

pio.renderers.default = 'notebook'     


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [4]:
GREY   = '#B8B8B8'
ACCENT = '#D55E00'   
CONT_COLORS = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#009E73'}

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach:** I will calculate the population growth of each continent compared to 1970 (set as 100). This makes it easier to compare their growth over time. The fastest-growing continent will be highlighted in a different colour.

**Justification:** A line chart clearly shows population changes over the years. Using grey lines with one highlight colour keeps the chart simple and easy to understand.

In [7]:
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100
cont['Label'] = np.where(cont['Year'] == 2022, cont['Continent'], '')

africa = cont[(cont['Continent'] == 'Africa') & (cont['Year'] == 2022)]['Index'].iloc[0] / 100

fig = px.line(cont, x='Year', y='Index', color='Continent', text='Label',
              color_discrete_map={'Africa': ACCENT, 'Asia': GREY, 'Europe': GREY,
                                  'North America': GREY, 'South America': GREY, 'Oceania': GREY},
              template='plotly_white',
              title=f'Africa is growing fastest: it is now {africa:.1f}x its 1970 size')
fig.update_traces(mode='lines+text', textposition='middle right')
fig.update_xaxes(title=None, showgrid=False, range=[1970, 2030])
fig.update_yaxes(title='Population (1970 = 100)', showgrid=False)
fig.update_layout(showlegend=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach**: I will calculate the population difference between 2022 and 1970, select the top 15 countries, and highlight African countries in a different colour.

**Justification**: A horizontal bar chart makes it easy to compare countries and read their names. Highlighting Africa also makes it easier to identify how many African countries are in the top 15.

In [9]:
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').copy()
top15['Added (M)'] = top15['Added'] / 1e6
top15['Group'] = np.where(top15.Continent == 'Africa', 'Africa', 'Other continents')
n_africa = (top15.Continent == 'Africa').sum()

fig = px.bar(top15, x='Added (M)', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa': ACCENT, 'Other continents': GREY},
             text='Added (M)', template='plotly_white',
             title=f'{n_africa} of the 15 countries that added the most people since 1970 are in Africa')
fig.update_traces(texttemplate='%{text:.0f}M', textposition='outside', cliponaxis=False)
fig.update_yaxes(title=None, categoryorder='total ascending')
fig.update_xaxes(visible=False)
fig.update_layout(legend=dict(title=None, orientation='h', y=-0.05, x=0), margin=dict(r=60))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach:** I will use a log scale for both variables and add a trend line to show their relationship. I will also label the countries that are furthest from the trend line.

**Justification:** A scatter plot is best for comparing two numerical variables. Different colours will represent continents, while labels will help identify countries that do not follow the general pattern.

In [10]:
fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 hover_name='Country', log_x=True, log_y=True,
                 template='plotly_white',
                 title='Bigger land area does not mean more people: '
                       'city-states and huge empty countries break the pattern',
                 labels={'Area': 'Land area in km² (log scale)',
                         '2022 Population': '2022 population (log scale)'})

names = ['Monaco', 'Singapore', 'Vatican City', 'Greenland', 'Canada', 'Russia', 'China', 'India']
for name in names:
    row = df[df['Country'] == name]
    if len(row) > 0:
        fig.add_annotation(x=np.log10(row['Area'].iloc[0]),
                           y=np.log10(row['2022 Population'].iloc[0]),
                           text=name, showarrow=True, arrowhead=0, ax=30, ay=-25)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.update_layout(legend_title=None)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach:** I will use a log scale to show population density differences clearly. I will highlight and label the countries with the highest and lowest density values.

**Justification:** A box plot makes it easy to compare population density across continents and identify outliers. It also shows the spread of the data better than a simple bar chart.

In [16]:
top = df.groupby('Continent')['Density'].median().idxmax()
df['Highlight'] = np.where(df['Continent'] == top, top, 'Other')
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()

fig = px.box(df, x='Continent', y='Density', color='Highlight', points='outliers',
             hover_name='Country', log_y=True,
             category_orders={'Continent': order},
             color_discrete_map={top: ACCENT, 'Other': GREY},
             template='plotly_white',
             title=f'Density varies enormously inside every continent; {top} has the highest median')

fig.update_xaxes(title=None)
fig.update_yaxes(title='People per km² (log scale)', showgrid=False)
fig.update_layout(showlegend=False, boxmode='overlay')
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach**: Compute the percentage change and use a diverging palette centred on 0, so shrinking and growing countries look clearly different. PuOr is CVD-safe and avoids red/green. Cap the colour range at ±100% so a few extreme growers (Gulf states) don’t wash out the rest.

**Justification**: The question asks “where in the world”, so a map is needed. The diverging colour scale encodes the meaning (growth vs shrinkage).

In [17]:
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
df['Change capped'] = df['Change'].clip(-100, 100)
n_shrink = (df['Change'] < 0).sum()

fig = px.choropleth(df, locations='CCA3', color='Change capped',
                    hover_name='Country', hover_data={'Change': ':.1f', 'Change capped': False},
                    color_continuous_scale='PuOr', color_continuous_midpoint=0,
                    template='plotly_white',
                    title=f'{n_shrink} countries lost population between 2000 and 2022; most of the world grew')
fig.update_layout(coloraxis_colorbar_title='% change', margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach**: Use path=['Continent', 'Country'] so both levels appear in one figure. Rectangle area is the 2022 population, and colour is the continent.

**Justification**: A treemap shows part-to-whole across a hierarchy, which is exactly “continents and the countries within them”. It is the only chart type in the list that shows both levels at once.

In [19]:
total = df['2022 Population'].sum()
asia = df[df['Continent'] == 'Asia']['2022 Population'].sum() / total

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'],
                 values='2022 Population', color='Continent',
                 color_discrete_map={'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
                                     'North America': '#E69F00', 'South America': '#56B4E9',
                                     'Oceania': '#009E73', '(?)': 'white'},
                 template='plotly_white',
                 title=f'{asia:.0%} of the world lives in Asia, and China and India dominate it')
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Approach:** I will group the three smaller continents as "Rest of the world" and calculate each group's percentage for every year. Africa will be placed at the bottom and labelled clearly.

**Justification:** A 100% stacked bar chart makes it easy to compare how population shares change over time. It also keeps the chart simple and easy to read.

In [21]:
df_long['Group'] = np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                            df_long['Continent'], 'Rest of the world')

g = df_long.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
g['Share'] = g['Population'] / g.groupby('Year')['Population'].transform('sum') * 100
g['Year'] = g['Year'].astype(str)

africa70 = g[(g['Year'] == '1970') & (g['Group'] == 'Africa')]['Share'].iloc[0]
africa22 = g[(g['Year'] == '2022') & (g['Group'] == 'Africa')]['Share'].iloc[0]

fig = px.bar(g, x='Year', y='Share', color='Group', text_auto='.0f',
             category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
             color_discrete_map={'Africa': '#D55E00', 'Asia': '#0072B2',
                                 'Europe': '#CC79A7', 'Rest of the world': '#B8B8B8'},
             template='plotly_white',
             title=f"Africa's share of world population rose from {africa70:.0f}% to {africa22:.0f}% "
                   "while Europe's shrank")
fig.update_xaxes(title=None)
fig.update_yaxes(title='Share of world population (%)', showgrid=False)
fig.update_layout(legend_title=None)
fig.show()